Nom : Khadijetou Mohamed abe

Filière : GLCC

In [1]:
%%capture 

# pour exécuter l'algorithme ID3 sans réécrire le même code
# Ainsi, ce sera le modèle que nous allons utiliser dans notre random forest
%run ArbreDeDecission-ID3-C4_5-CART.ipynb

# 1. CODE SCRATCH (ID3)

In [2]:
import numpy as np


def bootstrap_sample(data):
    # """
    # C'est l'étape du "Bagging" (Bootstrap Aggregating) qui rend la forêt robuste.
    # Crée un échantillon bootstrap en tirant des données de l'original AVEC REMISE.
    # """
    # data.shape[0] nous donne le nombre total de lignes dans notre jeu de données.
    n_samples = data.shape[0]
    
    # np.random.choice choisit des nombres au hasard entre 0 et n_samples.
    # size=n_samples : on veut créer un nouveau tableau de la MÊME TAILLE que l'original.
    # replace=True : (avec remise) signifie qu'une même ligne peut être choisie plusieurs fois, 
    # et d'autres lignes ne seront jamais choisies. C'est ce qui crée la diversité !
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    
    # On utilise les indices générés aléatoirement pour extraire les lignes correspondantes de 'data'.
    return data[indices]


def random_forest_train(data, feature_names , max_features, n_trees=4 ):
    # """
    # La fonction va appeler directement la fonction id3 pour générer chaque n_trees arbre.
    # """
        
    # pour stocker tous les arbres générés.
    forest = []
    
    # On crée une liste des index de nos caractéristiques ( s'il y a 4 features, ça donne [0, 1, 2, 3]).
    initial_indices = list(range(data.shape[1] - 1))
    
    # Boucle pour créer chaque arbre
    for i in range(n_trees):
        # étape de Bagging: On crée un échantillon aléatoire avec remise à partir de nos données.
        # Chaque arbre verra un ensemble de données légèrement différent.
        sample = bootstrap_sample(data)
        
        # étape du Random Forest: On utilise notre fonction id3. On lui passe le paramètre 'max_features' 
        # pour qu'elle sélectionne aléatoirement les attributs à tester à chaque séparation.
        tree = id3(sample, initial_indices, feature_names, max_features=max_features)
        
        # On ajoute l'arbre généré.
        forest.append(tree)
        
    return forest




def predict_single_tree(tree, row):
    # Condition d'arrêt : on est sur une feuille
    if not isinstance(tree, dict): 
        return tree
    
    # On trouve la caractéristique racine et la valeur correspondante dans notre ligne
    feature_name = list(tree.keys())[0]
    value = row[features.index(feature_name)]
    branches = tree[feature_name]
    
    # .get() cherche 'value' dans le dictionnaire. 
    # S'il ne la trouve pas, il renvoie automatiquement le 2ème argument
    next_node = branches.get(value, list(branches.values())[0])
    
    # Appel récursif sur le noeud suivant
    return predict_single_tree(next_node, row)


def random_forest_predict(forest, X_test):
    y_pred = []
    
    for row in X_test:
        # Pour demander à chaque arbre de la forêt de faire une prédiction de chaque ligne.
        votes = [predict_single_tree(arbre, row) for arbre in forest]
        
        #On utilise np.unique pour avoir les classes et leurs comptes
        classes, counts = np.unique(votes, return_counts=True)
        
        # On trouve l'index du max et on ajoute directement la classe gagnante
        y_pred.append(classes[np.argmax(counts)])
        
    return np.array(y_pred)

## a. DATASET

In [3]:
data = np.array([
        # Revenu,   Dettes,   Historique Crédit, Statut Emploi, Approuvé
        ["Élevé",   "Faibles", "Bon",             "Employé",      "OUI"],
        ["Élevé",   "Élevées", "Bon",             "Employé",      "OUI"],
        ["Moyen",   "Faibles", "Bon",             "Employé",      "OUI"],
        ["Faible",  "Faibles", "Moyen",           "Indépendant",  "OUI"],
        ["Faible",  "Élevées", "Mauvais",         "Chômage",      "NON"],
        ["Faible",  "Élevées", "Mauvais",         "Indépendant",  "NON"],
        ["Moyen",   "Élevées", "Mauvais",         "Employé",      "NON"],
        ["Moyen",   "Faibles", "Moyen",           "Indépendant",  "OUI"],
        ["Élevé",   "Faibles", "Moyen",           "Indépendant",  "OUI"],
        ["Faible",  "Faibles", "Bon",             "Chômage",      "OUI"],
        ["Moyen",   "Élevées", "Bon",             "Chômage",      "NON"],
        ["Élevé",   "Élevées", "Mauvais",         "Indépendant",  "NON"],
        ["Faible",  "Faibles", "Mauvais",         "Employé",      "NON"],
        ["Moyen",   "Faibles", "Bon",             "Chômage",      "OUI"],
 ])

# Noms des caractéristiques
features = ["Revenu", "Dettes", "Historique Crédit", "Statut Emploi"]


## b. Exécution du code scratch

In [9]:
# On a par défaut n_trees=4 
# max_features=2 : à chaque noeud, on choisira au hasard 2 features parmi celles disponibles
forest_scratch = random_forest_train(data, features,2)

print("Random forest:")
k=1
for arbre in forest_scratch:
    print("\n arbre:",k)
    k+=1
    print_tree(arbre)
    

# On sépare X (features) et y (target) 
X_data = data[:, :-1]
y_data = data[:, -1]

# On fait des prédictions sur les mêmes données pour voir comment notre modèle se comporte
predictions_scratch = random_forest_predict(forest_scratch, X_data)



Random forest:

 arbre: 1
|---Revenu
│   |---Faible
│   │   |---Historique Crédit
│   │   │   |---Mauvais
│   │   │   │   |--- NON
│   │   │   |---Moyen
│   │   │   │   |--- OUI
│   |---Moyen
│   │   |---Statut Emploi
│   │   │   |---Chômage
│   │   │   │   |---Dettes
│   │   │   │   │   |---Faibles
│   │   │   │   │   │   |--- OUI
│   │   │   │   │   |---Élevées
│   │   │   │   │   │   |--- NON
│   │   │   |---Employé
│   │   │   │   |--- OUI
│   │   │   |---Indépendant
│   │   │   │   |--- OUI
│   |---Élevé
│   │   |--- OUI

 arbre: 2
|---Dettes
│   |---Faibles
│   │   |---Statut Emploi
│   │   │   |---Chômage
│   │   │   │   |--- OUI
│   │   │   |---Employé
│   │   │   │   |---Revenu
│   │   │   │   │   |---Faible
│   │   │   │   │   │   |--- NON
│   │   │   │   │   |---Élevé
│   │   │   │   │   │   |--- OUI
│   │   │   |---Indépendant
│   │   │   │   |--- OUI
│   |---Élevées
│   │   |---Statut Emploi
│   │   │   |---Chômage
│   │   │   │   |--- NON
│   │   │   |---Employé
│   │   │

## c. Code sklearn

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

# Sklearn a besoin de données numériques, pas de chaînes de caractères.
# On utilise OrdinalEncoder pour convertir "Élevé", "Moyen", "Faible" en 0, 1, 2 etc.
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X_data)

# On crée une instance du classifieur Random Forest
# n_estimators=4 : équivalent à notre n_trees
# criterion='entropy' : on utilise la même métrique que notre code scratch (id3)
# max_features='sqrt' : équivalente à nos 2 features (sqrt(4) = 2)
modele_sklearn_rf = RandomForestClassifier(n_estimators=4, criterion='entropy', max_features='sqrt')

# On entraîne le modèle sur les données encodées
modele_sklearn_rf.fit(X_encoded, y_data)

# On obtient les prédictions de sklearn
predictions_sklearn = modele_sklearn_rf.predict(X_encoded)


## d. Comparaison (scratch/sklearn)¶

In [10]:
print("--- Vraies valeurs ---")
print(y_data)

print("\n--- Prédictions du modèle Scratch ---")
print(predictions_scratch)

print("\n--- Prédictions du modèle Sklearn ---")
print(predictions_sklearn)

# Calcul de la précision pour notre modèle scratch
accuracy_scratch = np.sum(predictions_scratch == y_data) / len(y_data)

# Calcul de la précision pour le modèle sklearn (en utilisant une fonction de la bibliothèque)
accuracy_sklearn = accuracy_score(y_data, predictions_sklearn)

print("\n--- Comparaison des Performances ---")
print("Précision du modèle Scratch :", accuracy_scratch * 100,"%")
print("Précision du modèle Sklearn :", accuracy_sklearn * 100,"%")


--- Vraies valeurs ---
['OUI' 'OUI' 'OUI' 'OUI' 'NON' 'NON' 'NON' 'OUI' 'OUI' 'OUI' 'NON' 'NON'
 'NON' 'OUI']

--- Prédictions du modèle Scratch ---
['OUI' 'OUI' 'OUI' 'OUI' 'NON' 'NON' 'OUI' 'OUI' 'OUI' 'OUI' 'NON' 'NON'
 'NON' 'OUI']

--- Prédictions du modèle Sklearn ---
['OUI' 'OUI' 'OUI' 'OUI' 'NON' 'NON' 'NON' 'OUI' 'OUI' 'NON' 'NON' 'NON'
 'NON' 'OUI']

--- Comparaison des Performances ---
Précision du modèle Scratch : 92.85714285714286 %
Précision du modèle Sklearn : 92.85714285714286 %
